# 02 — Geography & deprivation (Stage 2)

Builds the geography and deprivation layers that the rest of the pipeline depends on:

- **LSOA21 boundaries** (ONS, BGC: Generalised Clipped) for the 12 NE LAs.
- **LAD24 boundaries** (ONS, BFE: Boundary Full Extent) for the 12 NE LAs.
- **LSOA21 PWCs** (ONS, total-population weighted; DEC-012).
- **IMD2025** (MHCLG File 1) and **IDACI** (File 3 sub-domain).
- **Census 2021 school-age population** (TS007a, ages 5–19) per LSOA21.
- **RUC21 / Urban–Rural** classification per LSOA21.

Plus the deferred **schools point-in-polygon validation** against LAD polygons (the Stage 1 plan moved this here once boundaries are loaded).

All raw downloads are registered in `data/manifest.csv` with SHA256.

## 0. Pre-flight

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd

from schools_sunbeds import audit, config, deprivation, geography, population

config.ensure_dirs()
audit.verify_manifest()

TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
GEO_DIR = config.DATA_RAW / "geography" / TODAY
IMD_DIR = config.DATA_RAW / "imd2025" / TODAY
CEN_DIR = config.DATA_RAW / "census2021" / TODAY
for d in (GEO_DIR, IMD_DIR, CEN_DIR):
    d.mkdir(parents=True, exist_ok=True)
TODAY

## 1. Boundary layers — LSOA21 (BGC), LAD24 (BFE), LSOA21 PWC

All from ONS Open Geography Portal feature services. Downloads are filtered by NE bounding box for the LSOA layers; the LAD layer filters server-side by LAD code list.

In [ ]:
lsoa_path = geography.fetch_lsoa21_boundaries_ne(GEO_DIR)
lad_path = geography.fetch_lad_boundaries_ne(GEO_DIR)
pwc_path = geography.fetch_lsoa21_pwc(GEO_DIR)

for p, src_url, lic in (
    (lsoa_path, geography.ONS_LSOA21_BGC_URL, "Open Government Licence v3.0"),
    (lad_path, geography.ONS_LAD24_BFE_URL, "Open Government Licence v3.0"),
    (pwc_path, geography.ONS_LSOA21_PWC_URL, "Open Government Licence v3.0"),
):
    audit.register_raw_file(p, source_url=src_url, licence=lic, notes="ONS Open Geography Portal")
    print("registered:", p.relative_to(config.DATA_ROOT))

## 2. IMD2025 (File 1) and IDACI (File 3) from MHCLG

In [ ]:
imd1_path = deprivation.fetch_imd2025_file1(IMD_DIR)
imd3_path = deprivation.fetch_imd2025_file3(IMD_DIR)

for p, src_url in (
    (imd1_path, deprivation.IMD2025_FILE1_URL),
    (imd3_path, deprivation.IMD2025_FILE3_URL),
):
    audit.register_raw_file(p, source_url=src_url, licence="Open Government Licence v3.0", notes="MHCLG IoD2025")
    print("registered:", p.relative_to(config.DATA_ROOT))

## 3. Census 2021 (TS007a) and RUC21

In [ ]:
census_zip = population.fetch_census_ts007a_zip(CEN_DIR)
ruc_path = population.fetch_lsoa21_ruc_ne(GEO_DIR)

audit.register_raw_file(
    census_zip,
    source_url=population.CENSUS_TS007A_ZIP_URL,
    licence="Open Government Licence v3.0",
    notes="ONS Census 2021 TS007a (age 5y bands)",
)
audit.register_raw_file(
    ruc_path,
    source_url=population.ONS_LSOA21_RUC_URL,
    licence="Open Government Licence v3.0",
    notes="LSOA21 RUC21 attributes (no geometry needed)",
)
print("registered all Stage-2 raw inputs.")

## 4. Build the assembled LSOA21 NE table

Steps:

1. Load LAD24 polygons; the bbox-fetched LSOA21s are then attributed to a LAD via centroid-in-polygon spatial join, and we keep only LSOAs whose centroid sits in one of the 12 NE LAs.
2. Left-join IMD, IDACI, school-age population, and RUC.
3. Persist as a single GeoPackage that downstream stages read.

In [ ]:
lad = geography.load_lad(lad_path)
lsoa = geography.load_lsoa21(lsoa_path)
lsoa = geography.assign_lsoa_to_lad(lsoa, lad)

lsoa_ne = lsoa.loc[lsoa["lad_code"].isin(config.LA_CODES_NE)].reset_index(drop=True)
print(f"LSOA21 in NE (centroid-in-NE-LAD): {len(lsoa_ne):,}")
lsoa_ne["lad_code"].value_counts().sort_index().to_frame("lsoa_count")

In [ ]:
imd = deprivation.load_imd2025_main(imd1_path)
idaci = deprivation.load_imd2025_idaci(imd3_path)
pop = population.load_lsoa_school_age_population(census_zip)
ruc = population.load_lsoa21_ruc(ruc_path, lsoa_codes=lsoa_ne["lsoa21cd"])

joined = (
    lsoa_ne
    .merge(imd, on="lsoa21cd", how="left")
    .merge(idaci, on="lsoa21cd", how="left")
    .merge(pop, on="lsoa21cd", how="left")
    .merge(ruc, on="lsoa21cd", how="left")
)

audit_cols = {
    "missing_imd_decile": joined["imd_decile"].isna().sum(),
    "missing_idaci_decile": joined["idaci_decile"].isna().sum(),
    "missing_pop_school_age": joined["pop_school_age"].isna().sum(),
    "missing_urban_rural": joined["urban_rural"].isna().sum(),
}
print("Join completeness:")
for k, v in audit_cols.items():
    print(f"  {k:>22s}: {v}")

joined[["lsoa21cd", "lad_code", "imd_decile", "imd_quintile", "idaci_decile", "pop_school_age", "urban_rural"]].head()

## 5. PWCs filtered to NE

The PWC fetch was bbox-filtered, so we narrow further to the LSOAs that survived the centroid-in-NE-LAD step.

In [ ]:
pwc_all = geography.load_lsoa21_pwc(pwc_path)
pwc_ne = pwc_all.loc[pwc_all["lsoa21cd"].isin(joined["lsoa21cd"])].reset_index(drop=True)
print(f"PWCs retained: {len(pwc_ne):,} of {len(pwc_all):,} bbox-fetched")
missing_pwc = set(joined["lsoa21cd"]) - set(pwc_ne["lsoa21cd"])
print(f"LSOAs missing a PWC: {len(missing_pwc)}")  # expect 0

## 6. Schools point-in-polygon validation (deferred from Stage 1)

We load the Stage 1 schools GeoPackage and validate each school's geometry against the LAD polygons. A clean run shows every school inside its declared LA polygon.

In [ ]:
schools_files = sorted(config.DATA_PROCESSED.glob("schools_ne_*.gpkg"))
schools_files = [p for p in schools_files if "sensitivity" not in p.name]
if not schools_files:
    raise RuntimeError("No Stage 1 schools GeoPackage found — run notebook 01 first.")
schools_path = schools_files[-1]
schools_gdf = gpd.read_file(schools_path)
print(f"Loaded {schools_path.name} ({len(schools_gdf):,} schools)")

pip = geography.validate_schools_in_la(schools_gdf, lad)
print(f"PIP audit:")
print(f"  agree    : {pip['agree'].sum():>5}")
print(f"  disagree : {(~pip['agree']).sum():>5}")
print(f"  null poly: {pip['polygon_la'].isna().sum():>5}")

pip_path = config.AUDIT_LOGS / f"schools_pip_{TODAY}.csv"
pip.to_csv(pip_path, index=False)
print(f"Audit log written: {pip_path.relative_to(config.REPO_ROOT)}")

## 7. Write the assembled outputs

- `data/processed/lsoa_ne_<date>.gpkg` (one row per NE LSOA21 with all attributes)
- `data/processed/lsoa_pwc_ne_<date>.gpkg` (PWCs)
- `data/processed/lad_ne_<date>.gpkg` (LAD polygons)

In [ ]:
out_lsoa = config.DATA_PROCESSED / f"lsoa_ne_{TODAY}.gpkg"
out_pwc = config.DATA_PROCESSED / f"lsoa_pwc_ne_{TODAY}.gpkg"
out_lad = config.DATA_PROCESSED / f"lad_ne_{TODAY}.gpkg"

joined.to_file(out_lsoa, driver="GPKG", layer="lsoa21_ne")
pwc_ne.to_file(out_pwc, driver="GPKG", layer="lsoa21_pwc_ne")
lad.to_file(out_lad, driver="GPKG", layer="lad_ne")

for out_path, inputs, notes in (
    (
        out_lsoa,
        (lsoa_path, lad_path, imd1_path, imd3_path, census_zip, ruc_path),
        "NE LSOA21 BGC + IMD2025 + IDACI + RUC21 + Census 2021 school-age population",
    ),
    (out_pwc, (pwc_path,), "NE LSOA21 population-weighted centroids (total-pop weighted)"),
    (out_lad, (lad_path,), "NE LAD24 BFE polygons"),
):
    audit.write_provenance_sidecar(
        audit.Provenance(
            output_path=out_path,
            inputs=inputs,
            notes=notes,
            random_seed=config.RANDOM_SEED,
        )
    )
    print("wrote:", out_path.name)

## 8. Headline NE summary

A quick credibility check: per-LA totals of LSOAs, school-age population, and IMD quintile distribution. Stages 7+ will turn this into the descriptive replication of Lorigan et al. (2025).

In [ ]:
summary = (
    joined.groupby("lad_code")
    .agg(
        n_lsoa=("lsoa21cd", "count"),
        pop_total=("pop_total", "sum"),
        pop_school_age=("pop_school_age", "sum"),
        median_imd_decile=("imd_decile", "median"),
    )
    .sort_index()
)
summary.index = summary.index.map(lambda c: f"{c} {config.LA_NAMES_NE[c]}")
summary.loc["TOTAL"] = summary.sum(numeric_only=True)
summary

In [ ]:
imd_quintile_share = (
    joined.groupby(["lad_code", "imd_quintile"]).size().unstack(fill_value=0)
)
imd_quintile_share = imd_quintile_share.div(imd_quintile_share.sum(axis=1), axis=0)
imd_quintile_share.index = imd_quintile_share.index.map(lambda c: config.LA_NAMES_NE[c])
imd_quintile_share.round(2)

## 9. Quick-look choropleth

Sanity check: plot the LSOA21 NE polygons coloured by IMD quintile. The most-deprived (quintile 1) should cluster in the conurbations.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 9))
joined.plot(
    ax=ax,
    column="imd_quintile",
    cmap="RdYlGn_r",
    legend=True,
    edgecolor="none",
    linewidth=0,
    legend_kwds={"label": "IMD2025 quintile (1 = most deprived)"},
)
lad.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.5)
ax.set_title(f"NE LSOA21 by IMD2025 quintile (n_lsoa={len(joined):,})")
ax.set_xlabel("Easting (m, BNG)")
ax.set_ylabel("Northing (m, BNG)")
plt.tight_layout()
plt.show()

## Done

Outputs:

- `data/processed/lsoa_ne_<date>.gpkg`
- `data/processed/lsoa_pwc_ne_<date>.gpkg`
- `data/processed/lad_ne_<date>.gpkg`
- `audit_logs/schools_pip_<date>.csv` (per-school PIP-validation report)

All raw inputs registered in `data/manifest.csv`. Stage 4 (`06_network_build.ipynb`) reads the LAD GeoPackage to clip the OSM extract; Stages 5+ read the LSOA + PWC GeoPackages.